# Time travel with a checkpointer

This notebook builds a tiny two-step story pipeline (topic -> draft -> final)
with a checkpointer attached, then uses `get_state`, `get_state_history`, and
`update_state` to inspect past checkpoints, replay the graph from an old one
unchanged, and fork it by rewriting the topic and re-running from that point
forward. `09_Persistence/Persistence_Guide.md` already introduces time travel as
a concept; this notebook is the runnable, step-by-step version of it.

Imports: the usual graph pieces, plus `InMemorySaver`, the checkpointer that makes time travel possible in the first place. Without a checkpointer there is no history to travel through.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.7)

## The graph: topic -> draft -> final

`draft_step` writes a one-sentence story opening for the topic. `polish_step` adds a one-sentence ending. Each step is its own checkpoint once a checkpointer is attached.

In [ ]:
class StoryState(TypedDict):
    topic: str
    draft: str
    final: str


def draft_step(state: StoryState) -> dict:
    response = llm.invoke(f"Write one short, silly story-opening sentence about {state['topic']}.")
    return {"draft": response.content}


def polish_step(state: StoryState) -> dict:
    response = llm.invoke(f"Write one short closing sentence that finishes this story:\n{state['draft']}")
    return {"final": f"{state['draft']} {response.content}"}

In [ ]:
story_graph = StateGraph(StoryState)
story_graph.add_node("draft_step", draft_step)
story_graph.add_node("polish_step", polish_step)
story_graph.add_edge(START, "draft_step")
story_graph.add_edge("draft_step", "polish_step")
story_graph.add_edge("polish_step", END)

story_app = story_graph.compile(checkpointer=InMemorySaver())
story_app

## Running the pipeline once

A `thread_id` is required so the checkpointer knows which conversation's history to save checkpoints under, the same as in `09_Persistence`.

In [ ]:
config = {"configurable": {"thread_id": "story-1"}}

original = story_app.invoke({"topic": "a lost umbrella", "draft": "", "final": ""}, config)
original

## `get_state`: the latest checkpoint

`get_state(config)` returns a snapshot of where the thread currently stands: its `.values` (the state dict) and `.next` (the nodes that would run next — empty once the graph has finished).

In [ ]:
current = story_app.get_state(config)
print(current.values)
print(current.next)

## `get_state_history`: every checkpoint, newest first

`get_state_history(config)` yields one snapshot per step that ran, starting from the most recent. Each snapshot has `.values` (state at that point), `.next` (what was about to run), and `.config` (which pins down that exact checkpoint — this is what you pass back into `invoke` or `update_state` to act on that specific point in time).

In [ ]:
history = list(story_app.get_state_history(config))

for snapshot in history:
    print(snapshot.next, "->", snapshot.values)

## Replaying from an old checkpoint

The snapshot whose `.next` is `('polish_step',)` is the checkpoint saved right after `draft_step` ran, before `polish_step` touched it. Passing that snapshot's `.config` to `invoke` with no new input (`None`) resumes the graph from exactly that point — it does not redo `draft_step`.

In [ ]:
before_polish = next(s for s in history if s.next == ("polish_step",))
print(before_polish.values)

In [ ]:
replayed = story_app.invoke(None, before_polish.config)
replayed

## Forking: rewrite state at a checkpoint, then continue

The snapshot whose `.next` is `('draft_step',)` is the checkpoint saved right after the input was received, before `draft_step` ran. `update_state` writes a new value into that checkpoint and returns a new config pointing at the resulting checkpoint. Invoking from that new config re-runs the graph forward from there — with the new topic, `draft_step` runs again and produces a different story than the original run.

In [ ]:
before_draft = next(s for s in history if s.next == ("draft_step",))
print(before_draft.values)

In [ ]:
forked_config = story_app.update_state(before_draft.config, {"topic": "a haunted toaster"})
forked = story_app.invoke(None, forked_config)
forked

## Comparing the two runs

The original run and the forked run share the same graph and the same checkpoint history up to `before_draft`, but diverge from there because the topic changed.

In [ ]:
print("original:", original["final"])
print("forked:  ", forked["final"])